In [1]:
from dateutil.parser import parse
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from sklearn.decomposition import PCA


In [2]:
# Step 1: Load the CSV file

df = pd.read_csv("filtered_nasdaq_data2.csv")


# Step 2: Preserve original date strings

df['Date_raw'] = df['Date'].astype(str).str.strip()


# Step 3: Define a robust date parsing function
# Uses dateutil's parse to handle mixed formats
# Returns pd.NaT if parsing fails

def try_parse(x):
    try:
        return parse(x)
    except:
        return pd.NaT


# Step 4: Apply the parsing function with a progress bar

tqdm.pandas()
df['Date'] = df['Date_raw'].progress_apply(try_parse)


# Step 5: Check parsing results

print(f"Successfully parsed dates: {df['Date'].notna().sum()}")
print(f"Failed to parse dates: {df['Date'].isna().sum()}")

print(df.loc[df['Date'].isna(), 'Date_raw'].unique()[:20])


# Step 7: Strip time, keep only date component
df['Date'] = df['Date'].dt.date

/var/folders/bh/8vlm7qp54dnf4_wrqpc5trsm0000gn/T/ipykernel_57251/3510822167.py:3: DtypeWarning: Columns (3,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("filtered_nasdaq_data2.csv")
100%|████████████████████████████████| 563285/563285 [00:09<00:00, 62396.47it/s]

Successfully parsed dates: 563285
Failed to parse dates: 0
[]


In [3]:
df['Date'].head()

0    2023-12-16
1    2023-12-16
2    2023-12-16
3    2023-12-16
4    2023-12-16
Name: Date, dtype: object

In [4]:
# Count the number of records for each stock symbol
ticker_counts = df['Stock_symbol'].value_counts()

# Print the count of news entries per ticker
print(ticker_counts)

Stock_symbol
GILD    12376
QQQ     11813
WFC     11301
MRK     10774
KO      10521
        ...  
UPS      3828
AMTD     3823
GDX      3742
PM       3685
XRT      3551
Name: count, Length: 92, dtype: int64


In [5]:
print("Start date:", df["Date"].min())
print("End date:", df["Date"].max())

Start date: 2009-04-14
End date: 2023-12-16


In [6]:
# Remove the 'Date_raw' column from the DataFrame
df.drop(columns=['Date_raw'], inplace=True)

In [7]:
# 2. Print the the first record (row 0)
row0 = df.loc[0]
print("Row 0 TextRank summary:")
print(row0['Textrank_summary'])
print("\nRow 0 Full article:")
print(row0['Article'])


Row 0 TextRank summary:
Accenture (ACN) issued a downbeat fiscal Q2 revenue outlook despite reporting higher fiscal Q1 adjusted earnings and revenue. In economic news, November housing starts jumped almost 15% sequentially to a 1.56 million annual rate, above expectations compiled by Bloomberg for 1.36 million and 1.359 million in October. UBS (UBS) gained more than 5% after activist investor Cevian Capital said it has taken a 1.3% stake in the Swiss bank for around 1.2 billion euros ($1.31 billion), saying that it "sees significant value potential" in UBS after its takeover of Credit Suisse.

Row 0 Full article:
Financial stocks were advancing in Tuesday afternoon trading, with the NYSE Financial Index rising nearly 1% and the Financial Select Sector SPDR Fund (XLF) ahead 0.7%.
The Philadelphia Housing Index was climbing 1.4%, and the Real Estate Select Sector SPDR Fund (XLRE) was up 0.8%.
Bitcoin (BTC-USD) was declining 1.2% to $42,142, and the yield for 10-year US Treasuries was dro

In [8]:
# 1. Restrict to the desired date range
df['Date'] = pd.to_datetime(df['Date'])
start_date = pd.to_datetime('2012-01-01')
end_date = pd.to_datetime('2020-12-31')
df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)].copy()

# 2. Define the list of desired tickers
target_symbols = [
    'ACN', 'AMGN', 'AMT', 'BAX', 'BSX', 'CAT', 'CI', 'CLX', 'CMCSA', 'CME', 'COF', 'COST', 'D',
    'DHR', 'DUK', 'EBAY', 'EOG', 'FDX', 'FSLR', 'GE', 'GILD', 'HUM', 'ICE', 'KKR', 'KO', 'MMM',
    'MRK', 'MS', 'NDAQ', 'NEE', 'NEM', 'NKE', 'NOC', 'NSC', 'PRU', 'QCOM', 'RL', 'SLB', 'TAP',
    'TJX', 'TRV', 'TSN', 'UPS', 'USB', 'V', 'WFC'
]

# 3. Filter the DataFrame using the 'Stock_symbol' column
df = df[df['Stock_symbol'].isin(target_symbols)].copy()

## 1. Text Feature Selection

### Data Summary

- Total rows: 211,165  
- Non-empty summaries: 115,494  
- Empty summaries: 95,671  
- Unique non-empty summaries: 109,784  
- Non-empty titles: 211,165  
- Unique non-empty titles: 170,470  
- Empty titles: 0  
- Unique stocks: 46  
- Date range: 2012-01-01 to 2020-12-31  

---

### Use `title` Instead of `Textrank_summary`

- **No Missing Values**  
  `title` is available for all rows, while nearly **45%** of `Textrank_summary` is missing.

- **Better Coverage**  
  Keeping `title` allows us to retain **all news articles**, avoiding data loss.

- **Lower Complexity**  
  `title` is short and concise, reducing **preprocessing and embedding time**.

- **Reasonable Trade-off**  
  Although `Textrank_summary` may contain more detail, `title` offers a better **balance of quality and practicality**.

---

### Final Choice

Use `title` as the main text input for NLP modeling and signal extraction.

In [9]:
print("Total rows:", len(df))
print("Non-empty summaries:", df['Textrank_summary'].notna().sum())
print("Empty summaries:", df['Textrank_summary'].isna().sum())
print("Unique non-empty summaries:", df['Textrank_summary'].dropna().nunique())
print("Number of unique stocks:", df['Stock_symbol'].nunique())
print("Date range:", df['Date'].min(), "→", df['Date'].max())
print("Empty title:", df['Article_title'].isna().sum())
print("Non-empty title:", df['Article_title'].notna().sum())
print("Unique non-empty title:", df['Article_title'].dropna().nunique())
original_empty_rows = df[df['Textrank_summary'].isna()]
print(f"Number of original rows with empty Textrank_summary: {len(original_empty_rows)}")

Total rows: 211165
Non-empty summaries: 115494
Empty summaries: 95671
Unique non-empty summaries: 109784
Number of unique stocks: 46
Date range: 2012-01-01 00:00:00 → 2020-12-31 00:00:00
Empty title: 0
Non-empty title: 211165
Unique non-empty title: 170470
Number of original rows with empty Textrank_summary: 95671


In [10]:
# Keep only relevant columns: Date, Stock_symbol, and Article_title
# Then remove duplicate rows based on Article_title (keep the first occurrence only)
df = df[['Date', 'Stock_symbol', 'Article_title']].drop_duplicates(subset=['Article_title'])

# Show the resulting DataFrame
print(df.head())

           Date Stock_symbol  \
1126 2020-12-29          ACN   
1127 2020-12-22          ACN   
1128 2020-12-20          ACN   
1129 2020-12-19          ACN   
1130 2020-12-18          ACN   

                                          Article_title  
1126             Accenture Reaches Analyst Target Price  
1127  A Look at Stock Market News From Boeing, Twitt...  
1128  Looking For The Best Stocks To Buy Before 2021...  
1129  These High-Yield Dividend Stocks Might Be in T...  
1130  BUZZ-U.S. STOCKS ON THE MOVE-Winnebago, BioTel...  


## 2. News Data Structure Explanation

Although certain time periods have no news coverage, **tickers are preserved** instead of being removed. This ensures:

- **Cross-sectional completeness**: all stocks remain in the universe regardless of temporary news absence.
- **Stable panel structure**: required for consistent feature alignment with market and fundamental data.

It is also observed that **multiple news articles** can appear for the same ticker on the same date. This is common for **high-profile or event-driven stocks**.

To address this, downstream feature engineering may:

- **Aggregate text** (e.g., concatenate or summarize headlines),
- **Average embeddings** across multiple articles,
- Or apply **sequence models** if article order is important.

Such steps ensure **robust and consistent text-based inputs** for modeling.

In [11]:
# Get all unique dates across the dataset
unique_dates = sorted(df['Date'].unique())

# Convert to a Series and compute the difference between consecutive dates
date_series = pd.Series(unique_dates)
date_diffs = date_series.diff().dropna()

# Find the maximum gap between two dates
max_gap = date_diffs.max()
max_gap_index = date_diffs.idxmax()

# Identify the start and end of the gap
start_date = date_series.iloc[max_gap_index - 1]
end_date = date_series.iloc[max_gap_index]

# Print results
print(f"Maximum news gap: {max_gap.days} days")
print(f"Between {start_date} and {end_date}")


Maximum news gap: 4 days
Between 2014-05-23 00:00:00 and 2014-05-27 00:00:00


In [12]:
# Group by (Date, Ticker) and count number of news entries
news_per_day = df.groupby(['Date', 'Stock_symbol']).size().reset_index(name='News_Count')

# Filter where count > 1 (i.e., multiple news on the same day)
duplicate_news = news_per_day[news_per_day['News_Count'] > 1]

# Show the first few results
print(duplicate_news.head(10))

         Date Stock_symbol  News_Count
4  2012-01-03          CAT           2
6  2012-01-03          MMM           2
7  2012-01-03          MRK           2
10 2012-01-03          TRV           2
15 2012-01-04          BAX           2
17 2012-01-04          CAT           3
20 2012-01-04          CME           2
21 2012-01-04          COF           2
22 2012-01-04          DUK           2
23 2012-01-04         EBAY           2


In [13]:
# Display the number of unique stock symbols in the DataFrame
print("Number of unique Stock_symbol:", df['Stock_symbol'].nunique())
len(df)

Number of unique Stock_symbol: 46


170470

## 3. FinBERT Embedding and Sentiment Extraction: Process Overview

This section applies a pre-trained FinBERT model to extract text features from news titles.

---

### Objective

Convert financial news titles into two types of model-ready features:

1. **CLS Embeddings**: Dense vectors capturing overall meaning.
2. **Sentiment Scores**: Probabilities for negative, neutral, and positive tone.

---

### Implementation

- A pre-trained FinBERT model is used for both embeddings and sentiment.
- Each title is tokenized and processed in batches for efficiency.
- The `[CLS]` token output from the last layer is used as the 768-dimension embedding.
- Sentiment scores give the likelihood of each class: negative, neutral, positive.
- If a title fails to process, it receives a zero vector and neutral score to keep data aligned.

---

### Output

Two files are created:

1. **CLS Embedding File** (`finbert_cls_embedding.csv`)  
   - Contains a 768-length vector for each title  
   - Includes date, ticker, and original title

2. **Sentiment Score File** (`finbert_sentiment_score.csv`)  
   - Includes three columns: `sent_neg`, `sent_neu`, `sent_pos`  
   - Also includes date, ticker, and title

In [14]:
'''
# Config
model_name = "yiyanghkust/finbert-tone"
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS GPU")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")
batch_size = 32

# Step 1: Load FinBERT
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_cls = AutoModel.from_pretrained(model_name).to(device).eval()
model_sent = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()

# Step 2: Load Data
texts = df['Article_title'].tolist()
dates = df['Date'].tolist()
tickers = df['Stock_symbol'].tolist()
print(len(texts))
print(len(dates))
print(len(tickers))

# Step 3: CLS Embedding & Sentiment Score
embeddings = []
sentiment_scores = []

print("Extracting embeddings and sentiment scores...")
for i in tqdm(range(0, len(texts), batch_size), desc="Batching"):
    batch_texts = texts[i:i + batch_size]
    batch_dates = dates[i:i + batch_size]
    batch_tickers = tickers[i:i + batch_size]

    try:
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)

        with torch.no_grad():
            # CLS embedding
            output_cls = model_cls(**inputs)
            cls_vec = output_cls.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_vec)

            # Sentiment classification
            output_sent = model_sent(**inputs)
            probs = torch.nn.functional.softmax(output_sent.logits, dim=1).cpu().numpy()
            sentiment_scores.extend(probs)

    except Exception as e:
        print(f"Batch {i} failed:", e)
        embeddings.extend([np.zeros(768)] * len(batch_texts))
        sentiment_scores.extend([[0, 1, 0]] * len(batch_texts))  # Default neutral

# Step 4: Save Embedding CSV
embedding_df = pd.DataFrame(embeddings, columns=[f'embed_{i+1}' for i in range(768)])
embedding_df['Date'] = dates
embedding_df['Stock_symbol'] = tickers
embedding_df['Article_title'] = texts
embedding_df.to_csv("finbert_cls_embedding.csv", index=False)

# Step 5: Save Sentiment Score CSV
sent_df = pd.DataFrame(sentiment_scores, columns=['sent_neg', 'sent_neu', 'sent_pos'])
sent_df['Date'] = dates
sent_df['Stock_symbol'] = tickers
sent_df['Article_title'] = texts
sent_df.to_csv("finbert_sentiment_score.csv", index=False)
'''

'\n# Config\nmodel_name = "yiyanghkust/finbert-tone"\nif torch.backends.mps.is_available():\n    device = torch.device("mps")\n    print("Using Apple MPS GPU")\nelif torch.cuda.is_available():\n    device = torch.device("cuda")\n    print("Using NVIDIA CUDA GPU")\nelse:\n    device = torch.device("cpu")\n    print("Using CPU")\nbatch_size = 32\n\n# Step 1: Load FinBERT\ntokenizer = AutoTokenizer.from_pretrained(model_name)\nmodel_cls = AutoModel.from_pretrained(model_name).to(device).eval()\nmodel_sent = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()\n\n# Step 2: Load Data\ntexts = df[\'Article_title\'].tolist()\ndates = df[\'Date\'].tolist()\ntickers = df[\'Stock_symbol\'].tolist()\nprint(len(texts))\nprint(len(dates))\nprint(len(tickers))\n\n# Step 3: CLS Embedding & Sentiment Score\nembeddings = []\nsentiment_scores = []\n\nprint("Extracting embeddings and sentiment scores...")\nfor i in tqdm(range(0, len(texts), batch_size), desc="Batching"):\n    

## 4. FinBERT Embedding Dimensionality Reduction via Rolling PCA

This section reduces the size of FinBERT CLS embeddings using PCA in a rolling, out-of-sample setup.

---

### Objective

To create smaller, low-dimensional versions of FinBERT embeddings that still keep key information, while avoiding future-looking bias.

---

### Method

1. **Input**  
   - CLS embeddings from `finbert_cls_embedding.csv` (768 features)

2. **Rolling Time Splits**  
   - For each test year from 2017 to 2020:
     - Use the previous 5 years (e.g., 2012–2016) as training data
     - Fit PCA on training data only
     - Apply the PCA model to transform the test-year data
   - This keeps the process realistic and avoids using future data

3. **PCA Settings**  
   - Components: 10  
   - PCA is fit only once per time window  
   - Both train and test sets are transformed

4. **Output Columns**  
   - `pca_1` to `pca_10`: top 10 principal components  
   - Additional columns: `Date`, `Stock_symbol`, `Article_title`, `Test_Year`

---

In [15]:
'''
# Load CLS Embedding
embedding_df = pd.read_csv("finbert_cls_embedding.csv")
embedding_df['Date'] = pd.to_datetime(embedding_df['Date'])

# Config
pca_components = 10
all_pca_dfs = []

# Rolling PCA on Train + Test
for test_year in range(2017, 2021):
    print(f"\n=== Processing Test Year: {test_year} ===")
    train_start = f"{test_year - 5}-01-01"
    train_end   = f"{test_year - 1}-12-31"
    test_start  = f"{test_year}-01-01"
    test_end    = f"{test_year}-12-31"

    train_mask = (embedding_df['Date'] >= pd.to_datetime(train_start)) & \
                 (embedding_df['Date'] <= pd.to_datetime(train_end))
    test_mask = (embedding_df['Date'] >= pd.to_datetime(test_start)) & \
                (embedding_df['Date'] <= pd.to_datetime(test_end))

    train_df = embedding_df[train_mask].reset_index(drop=True)
    test_df  = embedding_df[test_mask].reset_index(drop=True)

    if test_df.empty or train_df.empty:
        print(f"Skipping year {test_year} due to empty split")
        continue

    print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

    # Fit PCA on train
    pca = PCA(n_components=pca_components, random_state=42)
    pca.fit(train_df.iloc[:, :768])

    # Transform both train and test
    for df_part in [train_df, test_df]:
        X_pca = pca.transform(df_part.iloc[:, :768])
        pca_df = pd.DataFrame(X_pca, columns=[f'pca_{i+1}' for i in range(pca_components)])
        pca_df['Date'] = df_part['Date'].values
        pca_df['Stock_symbol'] = df_part['Stock_symbol'].values
        if 'Article_title' in df_part.columns:
            pca_df['Article_title'] = df_part['Article_title'].values
        pca_df['Test_Year'] = test_year
        all_pca_dfs.append(pca_df)

# Save Final PCA File
final_pca_embedding_df = pd.concat(all_pca_dfs).reset_index(drop=True)
final_pca_embedding_df.to_csv("finbert_pca_embedding.csv", index=False)
print("Saved to finbert_pca_embedding.csv")
print(final_pca_embedding_df.head())
'''

'\n# Load CLS Embedding\nembedding_df = pd.read_csv("finbert_cls_embedding.csv")\nembedding_df[\'Date\'] = pd.to_datetime(embedding_df[\'Date\'])\n\n# Config\npca_components = 10\nall_pca_dfs = []\n\n# Rolling PCA on Train + Test\nfor test_year in range(2017, 2021):\n    print(f"\n=== Processing Test Year: {test_year} ===")\n    train_start = f"{test_year - 5}-01-01"\n    train_end   = f"{test_year - 1}-12-31"\n    test_start  = f"{test_year}-01-01"\n    test_end    = f"{test_year}-12-31"\n\n    train_mask = (embedding_df[\'Date\'] >= pd.to_datetime(train_start)) &                  (embedding_df[\'Date\'] <= pd.to_datetime(train_end))\n    test_mask = (embedding_df[\'Date\'] >= pd.to_datetime(test_start)) &                 (embedding_df[\'Date\'] <= pd.to_datetime(test_end))\n\n    train_df = embedding_df[train_mask].reset_index(drop=True)\n    test_df  = embedding_df[test_mask].reset_index(drop=True)\n\n    if test_df.empty or train_df.empty:\n        print(f"Skipping year {test_year}

In [16]:
# Load CLS embedding vectors from FinBERT
embedding_df = pd.read_csv("finbert_cls_embedding.csv")
print(">>> CLS Embedding DataFrame")
print(embedding_df.head())

# Load sentiment classification probabilities (negative, neutral, positive)
sent_df = pd.read_csv("finbert_sentiment_score.csv")
print("\n>>> Sentiment Score DataFrame")
print(sent_df.head())

# Load PCA-transformed embeddings (dimension reduced version)
pca_df = pd.read_csv("finbert_pca_embedding.csv")
print("\n>>> PCA Embedding DataFrame")
print(pca_df.head())

>>> CLS Embedding DataFrame
    embed_1   embed_2   embed_3   embed_4   embed_5   embed_6   embed_7  \
0 -0.132446  0.139046 -0.839172 -0.564129  1.397813 -0.142544 -0.287961   
1 -0.565734  0.212083  0.143279  0.182640  0.818591 -2.350612  0.209790   
2  0.049447 -0.346054 -0.178877  0.514553  0.888836  0.146242 -0.004710   
3 -1.013002 -1.028331 -0.389833 -0.103774 -0.359087 -0.083437 -0.421305   
4  0.393600 -1.254520 -1.344065 -0.286685  0.771646 -0.914001  0.324705   

    embed_8   embed_9  embed_10  ...  embed_762  embed_763  embed_764  \
0  1.145060 -1.666254 -0.060976  ...   0.433366  -0.046135   0.673648   
1  0.732383 -1.055083 -0.296134  ...   0.939721   0.247746  -0.085601   
2  0.278886 -0.944754  0.983990  ...   0.190811  -0.523842   0.784063   
3  1.184521  0.076914  0.428666  ...  -0.429518  -0.124768  -0.463782   
4  0.654297 -0.400255  0.019608  ...   0.528867  -0.180349   1.126009   

   embed_765  embed_766  embed_767  embed_768        Date  Stock_symbol  \
0   0.6

## 5. Sentiment-Weighted Text Embedding Construction

This section builds a daily, stock-level text feature by combining reduced FinBERT embeddings with sentiment scores. The goal is to reflect both **headline meaning** and **sentiment strength**.

---

### Objective

Create a clear and compact text-based signal that combines:

- **Meaning of headlines** (via PCA-reduced FinBERT vectors)  
- **Confidence in sentiment** (via FinBERT sentiment scores)

---

### Inputs

- `finbert_pca_embedding.csv`: 10 PCA components from FinBERT CLS embeddings  
- `finbert_sentiment_score.csv`: FinBERT sentiment scores — `sent_neg`, `sent_neu`, `sent_pos`  
- Merged using: `(Date, Stock_symbol, Article_title)`

---

### Step 1: Sentiment Weight

For each article $i$, define sentiment weight:

$$
w_i = \max(\text{sent\_neg}_i, \text{sent\_neu}_i, \text{sent\_pos}_i)
$$

This shows how confident the model is about the article's tone, used as a **reliability weight**.

---

### Step 2: Apply Weight to Embedding

Each PCA component $k$ is scaled by the sentiment weight:

$$
\text{pca}_{k,i}^{\text{weighted}} = w_i \cdot \text{pca}_{k,i}
$$

---

### Step 3: Aggregate by Stock and Date

For each stock $s$ on day $t$, average all weighted embeddings:

$$
\text{PCA}_{k}^{(s, t)} = \frac{\sum\limits_{i \in \mathcal{A}_{s,t}} w_i \cdot \text{pca}_{k,i}}{\sum\limits_{i \in \mathcal{A}_{s,t}} w_i}
$$

Where $\mathcal{A}_{s,t}$ is the set of all articles for that stock and day.

---

### Output

- File: `finbert_weighted_pca.csv`  
- Format: One row per `(Date, Stock_symbol)`  
- Columns: `pca_1` to `pca_10` (sentiment-weighted)  

In [17]:
# Step 1: Load and preprocess
pca_df = pd.read_csv("finbert_pca_embedding.csv")
sent_df = pd.read_csv("finbert_sentiment_score.csv")

pca_df['Date'] = pd.to_datetime(pca_df['Date'])
sent_df['Date'] = pd.to_datetime(sent_df['Date'])

# Step 2: Merge on exact (Date, Stock_symbol, Article_title)
merged_df = pd.merge(
    pca_df,
    sent_df[['Date', 'Stock_symbol', 'Article_title', 'sent_neg', 'sent_neu', 'sent_pos']],
    on=['Date', 'Stock_symbol', 'Article_title'],
    how='inner'
)

# Step 3: Use max(sentiment) as weight
merged_df['weight'] = merged_df[['sent_neg', 'sent_neu', 'sent_pos']].max(axis=1)

# Step 4: Weight PCA vectors
pca_cols = [col for col in merged_df.columns if col.startswith('pca_')]
for col in pca_cols:
    merged_df[f"{col}_weighted"] = merged_df[col] * merged_df['weight']

# Step 5: Compute weighted average by Date + Stock_symbol

# Step 5a: Sum weighted PCA values
weighted_sum_df = merged_df.groupby(['Date', 'Stock_symbol'])[
    [f"{col}_weighted" for col in pca_cols]
].sum().reset_index()

# Step 5b: Sum of weights per group
total_weight_df = merged_df.groupby(['Date', 'Stock_symbol'])['weight'].sum().reset_index(name='total_weight')

# Step 5c: Merge and normalize
weighted_avg_df = pd.merge(weighted_sum_df, total_weight_df, on=['Date', 'Stock_symbol'])
for col in pca_cols:
    weighted_avg_df[col] = weighted_avg_df[f"{col}_weighted"] / weighted_avg_df['total_weight']

# Step 6: Final formatting
weighted_pca_df = weighted_avg_df[['Date', 'Stock_symbol'] + pca_cols]

# Step 7: Save to file
weighted_pca_df.to_csv("finbert_weighted_pca.csv", index=False)
print("Saved weighted PCA to: finbert_weighted_pca.csv")
print(weighted_pca_df.head())

Saved weighted PCA to: finbert_weighted_pca.csv
        Date Stock_symbol     pca_1     pca_2     pca_3     pca_4     pca_5  \
0 2012-01-01          TRV -4.734676 -5.290467 -2.997122 -1.206622  2.074021   
1 2012-01-02          DUK  1.235866 -1.476278 -4.346992  5.224970 -3.745594   
2 2012-01-03          AMT  3.227606 -8.669047  5.685975 -8.500712 -2.244938   
3 2012-01-03          BAX -4.420509  5.250082 -2.331389 -2.738147  7.152293   
4 2012-01-03          CAT  7.334293  4.847746  1.120658 -2.446168  0.800946   

      pca_6     pca_7     pca_8     pca_9    pca_10  
0 -2.922532  0.651546  0.614291 -2.151206  0.662416  
1 -4.895936 -0.468432  2.153559  0.210452  4.970186  
2 -1.236409 -0.295759  5.051218  2.509494  6.266296  
3  2.574961  1.614328 -2.495943 -1.053265  2.245766  
4  1.439327 -0.045322 -2.825652 -2.866553  2.438894  
